# 🏠 Nyaya-Jyoti: AI-Powered House Sale Agreement Generator

Welcome to **Nyaya-Jyoti**, an intelligent chatbot-based legal document generator that helps you **create a personalized Agreement for Sale of House** — effortlessly and without any legal expertise.

This tool is powered by **GPT-Neo** and **Semantic Matching AI** to automatically generate standard as well as special legal clauses. The agreement is built dynamically based on your responses.

---

### 📄 How it works:

1. **Upload your clause CSV** and the **Sale Agreement Template (.docx)**
2. **Answer simple questions** related to the vendor, purchaser, property details, price, and terms
3. **Optionally enter a special clause** (e.g., “furniture included”) — AI will generate a custom clause
4. **Watch your document get updated live**, clause by clause
5. **Download your final House Sale Agreement** in `.docx` format

---

### ⚠️ Instructions:

- Be ready to upload:
  - ✅ The clause prompt CSV file (for special clause generation)
  - ✅ A `.docx` Word template with placeholders like `[Vendor_Name]`, `[House_Number]`, `[Sale_Price]`, etc.
- Provide accurate information (e.g., names, dates, and amounts)
- Special clauses will be handled after the main agreement details are filled
- Your document will download automatically once completed

---

> 🛡️ This tool is part of the *Nyaya-Jyoti* project by Bennett University. It is a demo/prototype and not a substitute for legal counsel.



In [ ]:
# @title
# 🛠️ Install required packages
!pip install -q transformers torch pandas sentence-transformers python-docx

# 📦 Imports
import torch
import pandas as pd
import re
import docx
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util
from google.colab import files
from IPython.display import display, Markdown

# --- Load Clause Prompt Registry ---
uploaded = files.upload()
csv_filename = list(uploaded.keys())[0]
prompt_df = pd.read_csv(csv_filename)
prompt_df["parameters"] = prompt_df["parameters"].fillna("").astype(str)
prompt_df["parameters"] = prompt_df["parameters"].apply(
    lambda x: ", ".join(sorted(set(p.strip() for p in x.split(",") if p.strip().lower() != "nan")))
)

# --- Load GPT-Neo & Embedding Model ---
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model.resize_token_embeddings(len(tokenizer))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
instruction_embeddings = embedder.encode(prompt_df['instruction'].tolist(), convert_to_tensor=True)

# --- Clause Generation Utilities ---
def build_prompt(example_1, example_2, instruction):
    return (
        "You are a legal assistant specialized in drafting formal legal clauses.\n"
        f"Example 1:\nClause: {example_1}\nEndClause\n\n"
        f"Example 2:\nClause: {example_2}\nEndClause\n\n"
        f"Now, generate ONLY the legal clause for {instruction}, using formal legal language.\n"
        "Output only the text between the markers 'Clause:' and 'EndClause'.\n\nClause: "
    )

def generate_clause(prompt_text):
    input_ids = tokenizer.encode(prompt_text, return_tensors="pt").to(device)
    output_ids = model.generate(
        input_ids,
        max_length=300,
        temperature=0.35,
        top_k=50,
        top_p=0.85,
        repetition_penalty=1.2,
        do_sample=True,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id
    )
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    if "EndClause" in generated_text:
        return generated_text.split("Clause:")[1].split("EndClause")[0].strip()
    return generated_text.split("Clause:")[1].strip()

def fill_parameters_dynamic(clause_text, param_string):
    placeholders = set(re.findall(r"{(.*?)}", clause_text))
    defined_params = [p.strip() for p in str(param_string).split(',') if p.strip()]
    combined_params = sorted(placeholders.union(set(defined_params)))
    param_values = {}
    for param in combined_params:
        value = input(f"🧾 Please provide value for '{param}': ").strip()
        param_values[param] = value
    for param, value in param_values.items():
        clause_text = clause_text.replace(f"{{{param}}}", value)
    return clause_text, param_values

def find_best_match_semantic(user_input):
    user_embedding = embedder.encode(user_input, convert_to_tensor=True)
    cosine_scores = util.pytorch_cos_sim(user_embedding, instruction_embeddings)[0]
    best_idx = torch.argmax(cosine_scores).item()
    best_score = cosine_scores[best_idx].item()
    if best_score > 0.25:
        return prompt_df.iloc[best_idx]
    return None

# --- Upload Template ---
print("📂 Upload your Agreement for Sale of House DOCX template:")
uploaded_template = files.upload()
template_filename = list(uploaded_template.keys())[0]
doc = docx.Document(template_filename)

# --- Placeholder Explanations ---
explanations = {
    "Vendor_Name": "Enter the full name of the person selling the house",
    "Vendor_Father_Name": "Enter the name of the vendor's father",
    "Vendor_Address": "Enter the address of the vendor",
    "Purchaser_Name": "Enter the full name of the person buying the house",
    "Purchaser_Father_Name": "Enter the name of the purchaser's father",
    "Purchaser_Address": "Enter the address of the purchaser",
    "Location": "Enter the city or place where the agreement is signed",
    "Day": "Enter the numeric day of the month",
    "Month": "Enter the month (e.g., April)",
    "Year": "Enter the four-digit year",
    "House_Number": "Enter the house number being sold",
    "Road_Name": "Enter the road name or locality of the house",
    "Sale_Price": "Enter the agreed sale price of the house",
    "Earnest_Money": "Enter the earnest money amount paid by the purchaser",
    "Earnest_Money_Date": "Enter the date on which earnest money was paid",
    "Sale_Period": "Enter the number of months within which sale should be completed",
    "Advocate_Report_Days": "Enter days within which purchaser will report on title deeds",
    "Refund_Days": "Enter number of days in which earnest money will be refunded if title not clear",
    "Refund_Interest_Days": "Enter number of days after which interest will apply on unrefunded earnest money",
    "Interest_Rate": "Enter the interest rate payable on delayed refund",
    "Liquidated_Damages": "Enter the amount to be paid by vendor as liquidated damages for breach",
    "Property_Description": "Provide a description of the property as referred in the schedule",
    "Special_Clauses": "Special terms or clauses if any (auto-filled below)"
}

# --- Fill Placeholders First ---
print("\n📋 Please answer the following questions to populate the agreement:")
user_inputs = {}
for placeholder, explanation in explanations.items():
    if placeholder == "Special_Clauses":
        continue
    value = input(f"🖋 {explanation}: ").strip()
    user_inputs[placeholder] = value
    for para in doc.paragraphs:
        if f"[{placeholder}]" in para.text:
            original_text = para.text
            para.text = para.text.replace(f"[{placeholder}]", value)
            display(Markdown(f"**📄 Updated Line:**\n\n`Before:` {original_text}\n\n`After:` {para.text}"))

# --- Special Clause Prompt Comes AFTER Placeholder Filling ---
special_clause_text = ""
print("\n📄 All standard fields are now filled.")
special_query = input("\n💬 Do you have any special clause to include (e.g., 'furniture included', 'pet living')?\n📨 Special Request (leave blank if none): ").strip()
if special_query:
    match_row = find_best_match_semantic(special_query)
    if match_row is not None:
        prompt = build_prompt(match_row['example_1'], match_row['example_2'], match_row['instruction'])
        raw_clause = generate_clause(prompt)
        special_clause_text, _ = fill_parameters_dynamic(raw_clause, match_row.get('parameters', ''))
    else:
        special_clause_text = "No special terms were included upon mutual agreement of the parties."

# --- Insert Special Clause in Template ---
if special_clause_text:
    for para in doc.paragraphs:
        if "[Special_Clauses]" in para.text:
            para.text = para.text.replace("[Special_Clauses]", special_clause_text)
            display(Markdown(f"**📄 Inserted Special Clause:** {para.text}"))
            break

# --- Save Final Agreement ---
output_file = "completed_house_sale_agreement.docx"
doc.save(output_file)
files.download(output_file)
print(f"\n✅ Agreement saved as: {output_file}")


Saving Verified_50_House_Sale_Clauses_NAMED.csv to Verified_50_House_Sale_Clauses_NAMED (1).csv
📂 Upload your Agreement for Sale of House DOCX template:


Saving agreement_for_sale_of_house_updated.docx to agreement_for_sale_of_house_updated (1).docx

📋 Please answer the following questions to populate the agreement:
🖋 Enter the full name of the person selling the house: Sunita


**📄 Updated Line:**

`Before:` This Agreement of sale made at [Location] on this [Day] day of [Month], [Year], between [Vendor_Name], son of [Vendor_Father_Name], resident of [Vendor_Address], hereinafter called the vendor of the ONE PART and [Purchaser_Name], son of [Purchaser_Father_Name], resident of [Purchaser_Address], hereinafter called the purchaser of the OTHER PART.

`After:` This Agreement of sale made at [Location] on this [Day] day of [Month], [Year], between Sunita, son of [Vendor_Father_Name], resident of [Vendor_Address], hereinafter called the vendor of the ONE PART and [Purchaser_Name], son of [Purchaser_Father_Name], resident of [Purchaser_Address], hereinafter called the purchaser of the OTHER PART.

**📄 Updated Line:**

`Before:` Signed and delivered by Shri [Vendor_Name],

`After:` Signed and delivered by Shri Sunita,

🖋 Enter the name of the vendor's father: Rajpal


...


📄 All standard fields are now filled.

💬 Do you have any special clause to include (e.g., 'furniture included', 'pet living')?
📨 Special Request (leave blank if none): furniture included


**📄 Inserted Special Clause:** 13. Special Terms and Conditions

    The vendor shall provide an inventory and condition report of all furniture sold with the property.

Schedule above referred to:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Agreement saved as: completed_house_sale_agreement.docx
